# Этап 2 — единая портфельная среда, без обучения PPO
**При подготовке notebook не выполнялся.** Результаты появятся только после запуска в Colab.
Прочитайте `docs/STAGE2_VALIDATION.md`: это синтетическая adjusted-price модель,
не номинальная share/dividend бухгалтерия. Решение до Open t, сделка по synthetic Open t,
reward после Close t. Исходные таблицы Stage 1 не меняются и не скачиваются заново.
Выберите CPU, Runtime Version **2026.07 (Python 3.12)**. Сохраняйте все выводы и ошибки.

## 1. Получение кода из GitHub и PROJECT_ROOT
`GIT_REF` может быть веткой, тегом или полным commit SHA. Для первого запуска
допустим `main`; для воспроизведения используйте SHA из отчёта успешного запуска.
Каждое выполнение этой ячейки создаёт новый checkout, не перезаписывая предыдущий.
Для приватного репозитория задайте `USE_GITHUB_SECRET=True` и добавьте в Colab Secrets
секрет `GITHUB_TOKEN` с правом чтения Contents. Токен не включается в URL или артефакты.
Версия notebook должна соответствовать выбранным исходникам.
Опубликуйте Stage 2 в GitHub. Задайте GIT_REF нового commit и SOURCE_RUN_ID подтверждённого Stage 1 каталога в Drive.

In [ ]:
import sys
import os
import re
import json
import subprocess
import tempfile
from pathlib import Path

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Запуск разрешён только в Google Colab, не в локальном runtime") from error
if not Path("/content").is_dir() or "COLAB_RELEASE_TAG" not in os.environ:
    raise RuntimeError("Нужна управляемая сессия Google Colab с каталогом /content")
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError("Нужен Python 3.11/3.12. Выберите Runtime > Change runtime type > Runtime Version 2026.07 (Python 3.12).")

GITHUB_REPOSITORY = "Go9shi/Science_Paper"
GIT_REF = "main"  # Для повторения замените на git_commit из metadata.json.
USE_GITHUB_SECRET = False
CHECK_RL_COMPATIBILITY = True  # Import/ABI check only; never training.
SOURCE_RUN_ID = "20261107T154124Z-9f1f7c1e"  # Existing complete Stage 1 run; no download.
NOTEBOOK_REVISION = "stage1-colab-env-v3"
REPO_URL = f"https://github.com/{GITHUB_REPOSITORY}.git"
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", GITHUB_REPOSITORY):
    raise ValueError("Ожидается repository в формате owner/name")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._/-]*", GIT_REF):
    raise ValueError("Некорректный GIT_REF")
if not re.fullmatch(r"[A-Za-z0-9_-]+", SOURCE_RUN_ID):
    raise ValueError("Некорректный SOURCE_RUN_ID")

REPO_ROOT = Path(tempfile.mkdtemp(prefix="Science_Paper-stage1-", dir="/content"))
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
with tempfile.TemporaryDirectory(prefix="stage1-auth-", dir="/content") as auth_dir:
    if USE_GITHUB_SECRET:
        from google.colab import userdata
        try:
            git_env["STAGE1_GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
        except Exception as error:
            raise RuntimeError("Разрешите доступ к Colab Secret GITHUB_TOKEN с правом чтения репозитория") from error
        if not git_env["STAGE1_GITHUB_TOKEN"]:
            raise RuntimeError("GITHUB_TOKEN пуст")
        askpass = Path(auth_dir) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['STAGE1_GITHUB_TOKEN'])\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        git_env["GIT_ASKPASS"] = str(askpass)
    try:
        subprocess.run(["git", "-c", "credential.helper=", "clone", "--quiet", "--no-checkout",
                        REPO_URL, str(REPO_ROOT)], env=git_env, check=True, capture_output=True, text=True, timeout=180)
        if re.fullmatch(r"[0-9a-fA-F]{7,40}", GIT_REF):
            # Full clone already contains reachable history; resolve short SHAs locally.
            resolved = subprocess.run(
                ["git", "-C", str(REPO_ROOT), "rev-parse", "--verify", f"{GIT_REF}^{{commit}}"],
                capture_output=True, text=True,
            )
            if resolved.returncode == 0:
                SOURCE_COMMIT = resolved.stdout.strip()
            elif len(GIT_REF) < 40:
                raise ValueError("Short SHA was not found or is ambiguous. Use the full 40-character commit SHA.")
            else:
                subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                               env=git_env, check=True, capture_output=True, text=True, timeout=180)
                SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        else:
            subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                           env=git_env, check=True, capture_output=True, text=True, timeout=180)
            SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
        raise RuntimeError("Не удалось получить код GitHub. Проверьте сеть, GIT_REF и доступ; для private используйте Secret.") from error
    finally:
        git_env.pop("STAGE1_GITHUB_TOKEN", None)
        git_env.pop("GIT_ASKPASS", None)

PROJECT_ROOT = REPO_ROOT / "research"
required = ["AGENTS.md", "src/__init__.py", "src/data.py", "src/volatility.py", "src/validation.py", "src/environment.py",
            "tests/test_data.py", "tests/test_volatility.py", "tests/test_validation.py", "tests/test_environment.py",
            "requirements-colab.txt", "requirements-rl-check.txt", "docs/METHODOLOGY.md", "docs/STAGE1_VALIDATION.md", "docs/COLAB_ENVIRONMENT.md",
            "notebooks/01_data_preparation.ipynb"]
if not CHECK_RL_COMPATIBILITY:
    raise RuntimeError("Stage 2 requires CHECK_RL_COMPATIBILITY=True")
required += ["src/portfolio.py", "src/environment_data.py", "src/portfolio_env.py", "src/env_validation.py",
             "tests/test_portfolio_env.py", "docs/STAGE2_VALIDATION.md", "notebooks/02_environment_validation.ipynb"]
missing = [name for name in required if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Commit {SOURCE_COMMIT} не содержит актуальный этап 1: {missing}. "
                            "Опубликуйте файлы и выберите содержащий их ref; локальные файлы не подменяют GitHub.")
source_notebook = json.loads((PROJECT_ROOT / "notebooks/01_data_preparation.ipynb").read_text(encoding="utf-8"))
if source_notebook["metadata"].get("stage1_revision") != NOTEBOOK_REVISION:
    raise RuntimeError("Версия открытого notebook не соответствует checkout; откройте notebook из выбранного commit")
print("GitHub:", GITHUB_REPOSITORY)
print("Resolved commit:", SOURCE_COMMIT)
print("PROJECT_ROOT:", PROJECT_ROOT)
stage2_source = json.loads((PROJECT_ROOT / "notebooks/02_environment_validation.ipynb").read_text(encoding="utf-8"))
if stage2_source["metadata"].get("stage2_revision") != "portfolio-env-v1":
    raise RuntimeError("Wrong Stage 2 notebook version; open notebook from selected commit")

## 2. Google Drive: подключение и проба записи/чтения
Авторизация производится интерфейсом Google. Проверяется именно MyDrive,
затем запись и чтение уникального probe-файла. Probe удаляется после проверки;
исходные данные и прошлые запуски не удаляются.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Drive не подключён: каталог MyDrive недоступен")
OUTPUT_ROOT = DRIVE_ROOT / "ppo_turnover_research" / "stage2"
PREFLIGHT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
PREFLIGHT_DIR = OUTPUT_ROOT / "preflight" / PREFLIGHT_ID
try:
    PREFLIGHT_DIR.mkdir(parents=True, exist_ok=False)
    probe = PREFLIGHT_DIR / "drive-probe.txt"
    probe_value = uuid4().hex
    probe.write_text(probe_value, encoding="utf-8")
    if probe.read_text(encoding="utf-8") != probe_value:
        raise OSError("Drive read-back mismatch")
    probe.unlink()
except OSError as error:
    raise RuntimeError(f"Нет надёжного чтения/записи Google Drive: {OUTPUT_ROOT}") from error
print("Drive read/write: OK")
print("Preflight logs:", PREFLIGHT_DIR)
STAGE1_RUN = DRIVE_ROOT / "ppo_turnover_research" / "stage1" / "runs" / SOURCE_RUN_ID
if not (STAGE1_RUN / "metadata.json").is_file():
    raise FileNotFoundError(f"Complete Stage 1 directory is unavailable: {STAGE1_RUN}")
print("Stage 1 source:", STAGE1_RUN)

## 3. Окружение env-v2: минимальная установка и диагностика
NumPy 2.0.2 и pandas 2.2.2 сохраняют базовые версии Colab 2026.07.
PyTorch не обновляется и не заменяется: его точный установленный build фиксируется
constraint-файлом. `CHECK_RL_COMPATIBILITY=True` добавляет только Gymnasium 1.2.0
и SB3 2.7.0 для проверки импортов и CPU-обмена NumPy↔Torch. Модели и торговая среда
не создаются, обучение не запускается. Используются base-пакеты без `[all]`/`[torch]` extras.
При несовпадении загруженных версий перезапустите сеанс и выполните сверху.
`python -m pip check` сохраняется и выводится полностью. Допустимы только два
документированных предупреждения неиспользуемых IPython/Jedi и Ibis/Toolz;
конфликты активных зависимостей и новые сообщения останавливают подготовку.
Эта версия окружения ещё не выполнялась в Colab — ниже код проверки, не результат.

In [ ]:
import importlib.metadata

# Read distribution metadata only: do not import NumPy/Pandas before pip.
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

before_install = {name: installed_version(name) for name in (
    "numpy", "pandas", "torch", "torchvision", "torchaudio", "google-colab",
    "ipython", "jedi", "ibis-framework", "toolz", "numba", "exchange-calendars",
)}
(PREFLIGHT_DIR / "versions-before-install.json").write_text(
    json.dumps(before_install, indent=2), encoding="utf-8")
install_args = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--report", str(PREFLIGHT_DIR / "pip-install-report.json"),
                "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
if CHECK_RL_COMPATIBILITY:
    if before_install["torch"] is None:
        raise RuntimeError("В runtime нет PyTorch. Выберите Colab 2026.07; автоматически скачивать/заменять torch не будем.")
    # Keep the exact preinstalled build, including its CUDA/CPU local suffix.
    constraint = PREFLIGHT_DIR / "runtime-constraints.txt"
    constraint.write_text(f"torch=={before_install['torch']}\n", encoding="utf-8")
    install_args += ["-r", str(PROJECT_ROOT / "requirements-rl-check.txt"), "-c", str(constraint)]
install = subprocess.run(install_args, text=True, capture_output=True)
(PREFLIGHT_DIR / "pip-install.log").write_text(install.stdout + "\n" + install.stderr, encoding="utf-8")
if install.returncode != 0:
    print(install.stdout[-4000:])
    print(install.stderr[-4000:])
    raise RuntimeError(f"Установка не прошла: {PREFLIGHT_DIR / 'pip-install.log'}; не обходите конфликт resolver.")
if CHECK_RL_COMPATIBILITY and installed_version("torch") != before_install["torch"]:
    raise RuntimeError("Версия PyTorch изменилась; проверка окружения остановлена")
print("Установка завершена без --upgrade; лог:", PREFLIGHT_DIR / "pip-install.log")

In [ ]:
import importlib
import importlib.metadata
import textwrap

requirements = {}
requirement_files = [PROJECT_ROOT / "requirements-colab.txt"]
if CHECK_RL_COMPATIBILITY:
    requirement_files.append(PROJECT_ROOT / "requirements-rl-check.txt")
for req_file in requirement_files:
    for line in req_file.read_text(encoding="utf-8").splitlines():
        if "==" in line and not line.lstrip().startswith("#"):
            package, version = line.strip().split("==")
            requirements[package] = version
module_names = {"stable-baselines3": "stable_baselines3"}
for package, version in requirements.items():
    installed = importlib.metadata.version(package)
    loaded = getattr(sys.modules.get(module_names.get(package, package)), "__version__", installed)
    if installed != version or loaded != version:
        raise RuntimeError(f"{package}: installed={installed}, loaded={loaded}, expected={version}. "
                           "Перезапустите сеанс Colab и выполните notebook сверху.")

project_env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT), "MPLBACKEND": "Agg"}
smoke_source = textwrap.dedent("""
import importlib, json
names = __MODULES__ + ['src.data', 'src.volatility', 'src.validation', 'src.environment', 'src.portfolio', 'src.environment_data', 'src.portfolio_env', 'src.env_validation']
modules = {name: importlib.import_module(name) for name in names}
from src.environment import dependency_closure
result = {
    'module_paths': {n: str(m.__file__) for n, m in modules.items()},
    'dependency_closure': dependency_closure(__ROOTS__),
    'future_stack': {'status': 'not_requested'},
}
if __CHECK_RL__:
    import numpy as np
    import torch
    from packaging.version import Version
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_checker import check_env
    if not (Version('2.3') <= Version(torch.__version__) < Version('3.0')):
        raise RuntimeError('PyTorch does not satisfy SB3 2.7.0')
    if Version(torch.__version__).base_version != '2.11.0':
        raise RuntimeError('Expected preinstalled PyTorch 2.11.0 for the selected 2026.07 baseline')
    array = np.array([1.0, 2.0], dtype=np.float32)
    recovered = torch.from_numpy(array).add(1).numpy()
    if not np.array_equal(recovered, np.array([2.0, 3.0], dtype=np.float32)):
        raise RuntimeError('NumPy/PyTorch CPU interoperability check failed')
    result['future_stack'] = {
        'status': 'passed', 'torch': torch.__version__, 'cuda_build': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(), 'numpy_torch_bridge': 'passed',
        'gymnasium': modules['gymnasium'].__version__,
        'stable_baselines3': modules['stable_baselines3'].__version__,
        'PPO_import': 'passed', 'model_created': False, 'training_performed': False,
        'project_environment_created': False,
    }
print(json.dumps(result, indent=2))
""")
smoke_source = (smoke_source.replace("__MODULES__", repr([module_names.get(p, p) for p in requirements]))
                .replace("__ROOTS__", repr(list(requirements) + (["torch"] if CHECK_RL_COMPATIBILITY else [])))
                .replace("__CHECK_RL__", repr(CHECK_RL_COMPATIBILITY)))
smoke = subprocess.run([sys.executable, "-c", smoke_source], cwd=PROJECT_ROOT,
                       env=project_env, capture_output=True, text=True)
(PREFLIGHT_DIR / "imports.log").write_text(smoke.stdout + "\n" + smoke.stderr, encoding="utf-8")
if smoke.returncode != 0:
    print(smoke.stderr[-4000:])
    raise RuntimeError(f"Ошибка import/ABI preflight; см. {PREFLIGHT_DIR / 'imports.log'}")
smoke_report = json.loads(smoke.stdout)

# Equivalent CLI command: python -m pip check (in this Colab kernel's Python).
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
(PREFLIGHT_DIR / "pip-check.txt").write_text(pip_check.stdout + "\n" + pip_check.stderr, encoding="utf-8")
print(pip_check.stdout or pip_check.stderr)
policy_source = (
    "import json, sys; from src.environment import classify_pip_check; "
    "payload=json.load(sys.stdin); print(json.dumps(classify_pip_check(payload['output'], payload['code'])))"
)
policy = subprocess.run([sys.executable, "-c", policy_source], input=json.dumps({
    "output": pip_check.stdout + "\n" + pip_check.stderr, "code": pip_check.returncode,
}), cwd=PROJECT_ROOT, env=project_env, capture_output=True, text=True, check=True)
classification = json.loads(policy.stdout)
environment = {
    "schema": 2, "profile": "env-v2-colab-2026.07", "rl_check_requested": CHECK_RL_COMPATIBILITY,
    "direct_versions": {p: importlib.metadata.version(p) for p in requirements},
    "versions_before_install": before_install,
    "project_imports": "passed", "subprocess_module_paths": smoke_report["module_paths"],
    "dependency_closure": smoke_report["dependency_closure"],
    "future_stack": smoke_report["future_stack"], "pip_check_policy": classification,
    "pip_check_returncode": pip_check.returncode, "drive_read_write": "passed",
    "python": sys.version, "colab_release_tag": os.environ["COLAB_RELEASE_TAG"],
}
(PREFLIGHT_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Фактические версии:", json.dumps(environment["direct_versions"], indent=2))
print("Dependency closure:", smoke_report["dependency_closure"]["status"])
print("Global pip check:", classification["status"])
print("Future stack:", json.dumps(environment["future_stack"], indent=2))
if smoke_report["dependency_closure"]["status"] != "passed" or classification["status"] == "blocked":
    raise RuntimeError(f"Неустранённый или новый конфликт; изучите {PREFLIGHT_DIR / 'environment.json'}")
print("Project dependency preflight passed; известные предупреждения, если есть, сохранены.")

## 4. Stage 2: конфигурация и происхождение
Общий бюджет/комиссии одинаковы для всех режимов. Числа ниже — параметры проверки механики,
не подбор по test. Будет использован только короткий заранее заданный сценарий на train.
Source hashes, Git bundle и runtime сохраняются в новом Stage 2 RUN_DIR.

In [ ]:
import platform
import shutil
from dataclasses import asdict

os.chdir(PROJECT_ROOT)
for name, module in list(sys.modules.items()):
    if name == "src" or name.startswith("src."):
        location = getattr(module, "__file__", None)
        if location and not Path(location).resolve().is_relative_to(PROJECT_ROOT.resolve()):
            raise RuntimeError("src loaded from another checkout; restart Colab and run from top")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
from src.data import sha256_file, write_json
from src.validation import pytest_summary
from src.environment_data import load_stage1_market, PRICE_MODEL
from src.portfolio_env import EnvConfig
from src.env_validation import deterministic_rollout, verify_stage2_run

COMMON = EnvConfig(initial_capital=1000., commission_bps=5., lambda_base=.001, adaptive_ratio_cap=5.)
COMMON.validate()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
source_paths = sorted(set([
    *PROJECT_ROOT.glob("src/*.py"), *PROJECT_ROOT.glob("tests/test_*.py"),
    *PROJECT_ROOT.glob(".agents/skills/*/SKILL.md"), PROJECT_ROOT / "AGENTS.md",
    PROJECT_ROOT / "requirements-colab.txt", PROJECT_ROOT / "requirements-rl-check.txt",
    *PROJECT_ROOT.glob("docs/*.md"), PROJECT_ROOT / "notebooks/02_environment_validation.ipynb",
]))
source_hashes = {}
for original in source_paths:
    relative = original.relative_to(PROJECT_ROOT)
    target = RUN_DIR / "provenance" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(original, target)
    source_hashes[relative.as_posix()] = sha256_file(original)
subprocess.run(["git", "-C", str(REPO_ROOT), "bundle", "create", str(RUN_DIR / "source.bundle"), "HEAD"], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "bundle", "verify", str(RUN_DIR / "source.bundle")], check=True, capture_output=True)
for original in PREFLIGHT_DIR.iterdir():
    if original.is_file():
        shutil.copy2(original, RUN_DIR / original.name)
write_json(RUN_DIR / "environment.json", environment)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True)
(RUN_DIR / "pip-freeze.txt").write_text(freeze.stdout, encoding="utf-8")
metadata = {
    "stage": 2, "schema": 1, "status": "started", "run_id": RUN_ID,
    "started_at_utc": datetime.now(timezone.utc).isoformat(), "git_commit": SOURCE_COMMIT,
    "python": sys.version, "platform": platform.platform(), "source_sha256": source_hashes,
    "stage1_directory": str(STAGE1_RUN), "common_config": asdict(COMMON),
    "price_model": PRICE_MODEL, "timing": "decision after previous close; synthetic open execution; current close reward",
    "training_performed": False, "tests": {"status": "not_run"},
}
write_json(RUN_DIR / "metadata.json", metadata)
print("Stage 2 RUN_ID:", RUN_ID)
print("Common config:", asdict(COMMON))

## 5. Все тесты, включая финансовые сценарии и оба env checker
Только синтетические данные, без сети и без обучения. Failure/error/skipped либо отсутствие
ключевых Stage 2 cases блокирует чтение реальных данных. Это первые реальные численные
проверки данной реализации; не считать их пройденными до выполнения ячейки.

In [ ]:
test_env = {**project_env, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}
result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", f"--junitxml={RUN_DIR / 'pytest.xml'}"],
                        cwd=PROJECT_ROOT, env=test_env, capture_output=True, text=True)
(RUN_DIR / "pytest.log").write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
print(result.stdout)
if result.returncode != 0:
    metadata.update(status="tests_failed", tests={"status": "failed", "exit_code": result.returncode})
    write_json(RUN_DIR / "metadata.json", metadata)
    raise RuntimeError(f"Tests failed; stop here. See {RUN_DIR / 'pytest.log'}")
try:
    summary = pytest_summary(RUN_DIR / "pytest.xml")
    import xml.etree.ElementTree as ET
    case_names = {case.get("name") for case in ET.parse(RUN_DIR / "pytest.xml").getroot().iter("testcase")}
    required_cases = {"test_gymnasium_and_sb3_checkers", "test_commission_is_financed_before_purchase",
                      "test_modes_have_identical_finances_and_distinct_reward_only"}
    if not required_cases.issubset(case_names):
        raise RuntimeError("Stage 2 financial/checker cases were not collected")
except Exception:
    metadata["status"] = "tests_failed"
    write_json(RUN_DIR / "metadata.json", metadata)
    raise
metadata["tests"] = {**summary, "exit_code": 0}
write_json(RUN_DIR / "metadata.json", metadata)
print("Actual test result:", summary)

## 6. Загрузка Stage 1 без изменения исходных файлов
**Принимается synthetic adjusted-price model**, описанный в STAGE2_VALIDATION.md.
Это не raw-share/cash-dividend ledger. При необходимости именно такого ledger остановитесь:
текущие данные не обеспечивают полную модель фактических выплат/номинальных share units.
Загрузчик проверяет prices/features/ref/split. Для validation/test выводится только покрытие.

In [ ]:
if metadata["tests"].get("status") != "passed":
    raise RuntimeError("Run all tests before loading market data")
try:
    datasets = {name: load_stage1_market(STAGE1_RUN, name, acknowledge_synthetic_prices=True)
                for name in ("train", "validation", "test")}
except Exception as error:
    metadata.update(status="data_failed", error=str(error))
    write_json(RUN_DIR / "metadata.json", metadata)
    raise
train = datasets["train"]
if len(train.assets) != 12:
    raise ValueError("This notebook expects the fixed 12-stock research universe")
coverage = {name: {"rows": len(data.features), "first": str(data.features.index.min().date()),
                   "last": str(data.features.index.max().date())} for name, data in datasets.items()}
metadata.update(stage1_metadata_sha256=train.provenance["stage1_metadata_sha256"],
                dataset_coverage=coverage, sigma_ref=train.sigma_ref, assets=list(train.assets))
write_json(RUN_DIR / "metadata.json", metadata)
print(json.dumps(coverage, indent=2))
print("Training-only sigma_ref:", train.sigma_ref)

## 7. Шесть фиксированных инструкций на первых шести train-сессиях
План одинаков для всех режимов: cash → акция 0 → удержание акции 0 → акция 1 →
равные веса 12 акций и cash → cash. Тикеры не выбираются по доходности, веса не
оптимизируются. Сценарий нужен для бухгалтерии, не для оценки эффективности PPO.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

n = len(train.assets)
cash_weights = np.r_[np.zeros(n), 1.]
first_weights = np.zeros(n + 1); first_weights[0] = 1.
second_weights = np.zeros(n + 1); second_weights[1] = 1.
uniform_weights = np.full(n + 1, 1 / (n + 1))
planned_weights = [cash_weights, first_weights, first_weights, second_weights, uniform_weights, cash_weights]
actions = [(2 * w - 1).astype(np.float32).tolist() for w in planned_weights]
metadata["action_schedule"] = actions
write_json(RUN_DIR / "metadata.json", metadata)
try:
    scenario, trades = deterministic_rollout(train, COMMON, actions)
except Exception as error:
    metadata.update(status="scenario_failed", error=str(error))
    write_json(RUN_DIR / "metadata.json", metadata)
    raise
display(scenario[["mode", "date", "nav_previous_close", "nav_pre_trade_open", "nav_post_trade_open",
                  "nav_close", "commission", "turnover", "r_net_log", "sigma_t", "lambda_t", "regularization_penalty", "reward"]])
display(trades.loc[trades["trade_notional"].abs() > 1e-8])  # Only display filter; complete ledger is saved.
print("All three modes reconciled under identical fixed actions; no policy was trained.")

## 8. Диагностики сценария и отдельное сохранение Stage 2
Совпадающие финансовые линии ожидаемы: действия одинаковы. Различие reward обусловлено
только штрафом, не обучением. Исходные Stage 1 файлы не переписываются.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(12, 11), constrained_layout=True)
for mode, rows in scenario.groupby("mode", sort=False):
    axes[0].plot(rows["date"], rows["portfolio_value"], marker="o", label=mode)
    axes[1].plot(rows["date"], rows["reward"].cumsum(), marker="o", label=mode)
    axes[2].plot(rows["date"], rows["turnover"], marker="o", label=mode)
for ax in axes:
    ax.legend(); ax.tick_params(axis="x", rotation=30)
axes[0].set(title="Fixed-action accounting check: synthetic portfolio NAV (not trained strategies)", ylabel="Synthetic NAV")
axes[1].set(title="Cumulative reward under identical actions", ylabel="Reward sum")
axes[2].set(title="Actual risky traded notional / pre-trade NAV", ylabel="Two-sided turnover")
fig.savefig(RUN_DIR / "diagnostics.png", dpi=140)
plt.show(); plt.close(fig)
scenario.to_csv(RUN_DIR / "scenario.csv", index=False)
trades.to_csv(RUN_DIR / "trades.csv", index=False)
train.adjusted_open.to_parquet(RUN_DIR / "adjusted_open.parquet")
train.adjusted_close.to_parquet(RUN_DIR / "adjusted_close.parquet")
for name, digest in source_hashes.items():
    if sha256_file(PROJECT_ROOT / name) != digest:
        raise RuntimeError(f"Source changed during run: {name}")
metadata["status"] = "prepared"
metadata["files_sha256"] = {p.relative_to(RUN_DIR).as_posix(): sha256_file(p)
    for p in sorted(RUN_DIR.rglob("*")) if p.is_file() and p.name not in {"metadata.json", "metadata.sha256"}}
write_json(RUN_DIR / "metadata.json", metadata)

## 9. Финальная read-back проверка
Повторно читает CSV/Parquet/Stage 1 источник, проверяет hashes и JUnit, воспроизводит
тот же короткий сценарий и финансовые равенства. Только затем записывает verified/complete.
Это проверка определённой синтетической модели, не реальных dividend cash-flows.

In [ ]:
try:
    report = verify_stage2_run(RUN_DIR)
except Exception as error:
    metadata.update(status="validation_failed", error=str(error))
    write_json(RUN_DIR / "metadata.json", metadata)
    raise
write_json(RUN_DIR / "validation_report.json", report)
metadata["files_sha256"]["validation_report.json"] = sha256_file(RUN_DIR / "validation_report.json")
metadata["status"] = "complete"
metadata["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
write_json(RUN_DIR / "metadata.json", metadata)
(RUN_DIR / "metadata.sha256").write_text(sha256_file(RUN_DIR / "metadata.json") + "  metadata.json\n", encoding="utf-8")
if (RUN_DIR / "metadata.sha256").read_text(encoding="utf-8").split() != [sha256_file(RUN_DIR / "metadata.json"), "metadata.json"]:
    raise RuntimeError("Metadata checksum failed after write/read-back")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("STAGE 2: COMPLETE (selected synthetic accounting model)")
print("Send metadata/sidecar, validation_report, pytest.xml/log, environment.json, pip-check.txt, scenario.csv, trades.csv and diagnostics.png")
print("PPO training performed:", report["training_performed"])